# 60 — Final Integrity Report
Distinguish readiness from proof.

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def normalise_date_col(df):
    for c in df.columns:
        if c.lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out = df.copy()
            out["date"] = pd.to_datetime(out[c], errors="coerce").dt.normalize()
            return out, c
    return df.copy(), None

def list_output_files():
    rows = []
    if not OUTPUTS.exists():
        return pd.DataFrame(columns=["path","size_bytes","sha256"])
    for p in sorted(OUTPUTS.rglob("*")):
        if p.is_file():
            rows.append({
                "path": str(p.relative_to(PROJECT_ROOT)),
                "size_bytes": p.stat().st_size,
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

run_cfg = load_yaml(CONFIGS / "run.yml")
phase_cfg = load_yaml(CONFIGS / "phase50.yml")
sites_df = pd.DataFrame(run_cfg.get("sites", []))
phase_sites = pd.DataFrame(phase_cfg.get("additional_sites", []))
if not phase_sites.empty:
    existing = set(sites_df.get("id", pd.Series(dtype=str)).astype(str).tolist()) if not sites_df.empty else set()
    phase_sites = phase_sites[~phase_sites["id"].astype(str).isin(existing)]
    sites_df = pd.concat([sites_df, phase_sites], ignore_index=True)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])
date_from = run_cfg.get("run", {}).get("date_from", "2025-01-01")
date_to = run_cfg.get("run", {}).get("date_to", "2025-01-31")

In [ ]:
step = "60_final_integrity_report"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

deps, _ = safe_read_csv("outputs/50_evidence_inventory/dependency_audit.csv")
adjud, _ = safe_read_csv("outputs/56_site_adjudication/site_adjudication.csv")
red, _ = safe_read_csv("outputs/57_contradictions_red_team_audit/contradiction_register.csv")

final = {
    "generated_at_utc": utcnow(),
    "repo_ready_for_forensic_review": bool(not adjud.empty and (adjud["verdict"] == "ready_for_review").any()),
    "undeniable_claim_supported": False,
    "reason_undeniable_false": "Scientific integrity requires falsifiability and uncertainty disclosure; this workflow can support audited, validated analysis but not an honest claim of undeniability.",
    "missing_dependency_count": int((~deps["exists"]).sum()) if not deps.empty else None,
    "contradiction_count": int(len(red)) if not red.empty else 0,
}
if not adjud.empty:
    final["site_verdicts"] = json.loads(adjud.to_json(orient="records"))
if not red.empty:
    final["contradictions"] = json.loads(red.to_json(orient="records"))

write_json(out / "final_integrity_report.json", final)

lines = [
    "# Final Integrity Report",
    "",
    f"Generated: {final['generated_at_utc']}",
    "",
    f"- Repo ready for forensic review: {final['repo_ready_for_forensic_review']}",
    f"- Undeniable claim supported: {final['undeniable_claim_supported']}",
    f"- Missing dependency count: {final['missing_dependency_count']}",
    f"- Contradiction count: {final['contradiction_count']}",
    "",
    "## Position",
    "This workflow is designed to increase auditability, integrity, provenance, and falsifiability.",
    "It should support defensible review, not overclaiming.",
]
(out / "final_integrity_report.md").write_text("\n".join(lines), encoding="utf-8")

manifest = manifest_base(step, [CONFIGS / "phase50.yml", CONFIGS / "evidence_thresholds.yml"])
for p in [out / "final_integrity_report.json", out / "final_integrity_report.md"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
write_json(out / "manifest.json", manifest)
display(pd.DataFrame([final]))